In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import joblib
import os

# ── Find dataset path automatically ───────────────
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# ── Load ──────────────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/gauriii09/gemma-4-data/gemmaopt_massive_dataset.csv')
print("Loaded:", df.shape)
print(df.head(3))

# ── Fix dataset ───────────────────────────────────
df.loc[df['Bottleneck'] == 'OOM (Crash)', 'Step_Time_ms'] = -1
df['OOM_Risk'] = (df['Bottleneck'] == 'OOM (Crash)').astype(int)
print("\nClass distribution:")
print(df['Bottleneck'].value_counts())

# ── Encode ────────────────────────────────────────
le_model  = LabelEncoder()
le_gpu    = LabelEncoder()
le_target = LabelEncoder()

df['Model_Enc']      = le_model.fit_transform(df['Model'])
df['GPU_Enc']        = le_gpu.fit_transform(df['GPU_Name'])
df['Precision_Enc']  = df['Precision'].map({'FP32': 0, 'FP16': 1})
df['Bottleneck_Enc'] = le_target.fit_transform(df['Bottleneck'])

print("\nTarget classes:", le_target.classes_)

# ── Features & Target ─────────────────────────────
features = [
    'Model_Enc', 'Batch_Size', 'Precision_Enc',
    'GPU_Enc', 'GPU_VRAM_GB', 'GPU_Bandwidth', 'Peak_VRAM_GB'
]

X = df[features]
y = df['Bottleneck_Enc']

# ── Train/Test Split ──────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("\nTrain size:", X_train.shape[0])
print("Test size: ", X_test.shape[0])

# ── Train XGBoost ─────────────────────────────────
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# ── Evaluate ──────────────────────────────────────
y_pred = model.predict(X_test)
acc = round(accuracy_score(y_test, y_pred) * 100, 1)
print("\n✅ Accuracy:", acc, "%")
print("\n", classification_report(
    y_test, y_pred,
    target_names=le_target.classes_
))

# ── Save ──────────────────────────────────────────
joblib.dump(model,     '/kaggle/working/gemmaopt_model.pkl')
joblib.dump(le_model,  '/kaggle/working/encoder_model.pkl')
joblib.dump(le_gpu,    '/kaggle/working/encoder_gpu.pkl')
joblib.dump(le_target, '/kaggle/working/encoder_target.pkl')

print("\n✅ All models saved to /kaggle/working/")
print("Files saved:")
for f in os.listdir('/kaggle/working'):
    print(" →", f)

/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/generation_config.json
/kaggle/input/datasets/gauriii09/gemma-4-data/gemmaopt_massive_dataset.csv
Loaded: (4608, 9)
      Model  Batch_Size Precision     GPU_Name  GPU_VRAM_GB  GPU_Bandwidth  \
0  resnet18           4      FP32  NVIDIA H100         80.0         3350.0   
1  resnet18           4      FP16  NVIDIA H100         80.0         3350.0   
2  

In [2]:
!pip install -q git+https://github.com/huggingface/transformers.git
import importlib
import transformers
importlib.reload(transformers)
print(transformers.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 97.3 MB/s eta 0:00:00:00:01
5.7.0.dev0


In [3]:
import os
import joblib
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ── Load XGBoost ───────────────────────────────────
model_xgb    = joblib.load('/kaggle/working/gemmaopt_model.pkl')
le_model_enc = joblib.load('/kaggle/working/encoder_model.pkl')
le_gpu_enc   = joblib.load('/kaggle/working/encoder_gpu.pkl')
le_target    = joblib.load('/kaggle/working/encoder_target.pkl')
print("✅ XGBoost loaded")

# ── Load Gemma 4 ───────────────────────────────────
GEMMA_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"
tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH)
gemma = AutoModelForCausalLM.from_pretrained(
    GEMMA_PATH,
    dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Gemma 4 loaded")

# ── Predict bottleneck ─────────────────────────────
def predict_bottleneck(model_name, batch_size, precision, gpu_name, vram_gb, bandwidth_gbs, peak_vram_gb):
    model_enc = le_model_enc.transform([model_name])[0]
    gpu_enc   = le_gpu_enc.transform([gpu_name])[0]
    prec_enc  = 1 if precision == 'FP16' else 0
    X = pd.DataFrame([{
        'Model_Enc'     : model_enc,
        'Batch_Size'    : batch_size,
        'Precision_Enc' : prec_enc,
        'GPU_Enc'       : gpu_enc,
        'GPU_VRAM_GB'   : vram_gb,
        'GPU_Bandwidth' : bandwidth_gbs,
        'Peak_VRAM_GB'  : peak_vram_gb
    }])
    pred       = model_xgb.predict(X)[0]
    prob       = model_xgb.predict_proba(X)[0]
    bottleneck = le_target.inverse_transform([pred])[0]
    confidence = round(max(prob) * 100, 1)
    return bottleneck, confidence

# ── Generate NVIDIA-level fix ──────────────────────
def generate_fix(model_name, batch_size, precision, gpu_name, vram_gb, bottleneck, confidence, peak_vram_gb, step_time_ms):
    chat = [
        {
            "role": "user",
            "content": f"""You are a senior GPU optimization engineer at NVIDIA with 10 years experience optimizing deep learning training kernels. You have shipped optimizations for PyTorch, CUDA, and cuDNN. You speak precisely and technically.

A researcher is training {model_name} with batch size {batch_size} in {precision} on {gpu_name} ({vram_gb}GB VRAM, {peak_vram_gb}GB peak usage, {step_time_ms}ms per step).

Your diagnostic system flagged: {bottleneck} at {confidence}% confidence.

Respond EXACTLY ONCE. No repetition. No filler.

[WHY]
Explain precisely what is happening at the hardware level. Reference memory bandwidth, CUDA kernel behavior, or compute utilization. Be specific to {model_name} on {gpu_name}. 3 sentences maximum.

[FIX]
Production-grade Python code fix. DO NOT suggest lowering the batch size. Write drop-in PyTorch code using advanced memory optimization like `torch.utils.checkpoint` (gradient checkpointing), `torch.autocast` (mixed precision), or fused optimizers. Do NOT write obvious comments like "import torch". Only comment on the exact GPU memory mechanism being altered.

[LESSON]
One sentence. Name the specific GPU optimization concept the researcher just learned."""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        chat,
        return_tensors="pt",
        return_dict=True, 
        add_generation_prompt=True
    ).to(gemma.device)

    with torch.no_grad():
        outputs = gemma.generate(
            **inputs, 
            max_new_tokens=400,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], 
        skip_special_tokens=True
    ).strip()

    if response.count('[WHY]') > 1:
        response = response[:response.index('[WHY]', 10)]

    return response

✅ XGBoost loaded


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✅ Gemma 4 loaded


In [4]:

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/generation_config.json
/kaggle/input/datasets/gauriii09/gemma-4-data/gemmaopt_massive_dataset.csv


In [5]:
import gradio as gr

def process_dashboard(model_name, batch_size, precision, gpu_name):
    # 1. Hardware specs mapping for the XGBoost engine
    if gpu_name == "NVIDIA T4":
        vram_gb, bandwidth_gbs = 16.0, 320.0
    elif gpu_name == "NVIDIA A100":
        vram_gb, bandwidth_gbs = 80.0, 2039.0
    else: # RTX 4090
        vram_gb, bandwidth_gbs = 24.0, 1008.0
        
    # peak VRAM and step time estimate logic
    peak_vram_gb = (batch_size / 512) * 14.2 
    step_time_ms = (batch_size / 512) * 11529

    # 2. Call XGBoost Engine
    bottleneck, confidence = predict_bottleneck(
        model_name, batch_size, precision, gpu_name, vram_gb, bandwidth_gbs, peak_vram_gb
    )

    # 3. Format Diagnostic Telemetry Box
    if "OOM" in bottleneck:
        diagnostic = f"🚨 **CRITICAL: {bottleneck} Predicted**\nConfidence: {confidence}%\nTarget: {gpu_name} | VRAM: {vram_gb}GB"
    else:
        diagnostic = f"🟢 **STABLE: {bottleneck}**\nConfidence: {confidence}%\nTarget: {gpu_name} | VRAM: {vram_gb}GB"

    # 4. Call Gemma 4 Engine for the NVIDIA-grade fix
    fix = generate_fix(
        model_name, batch_size, precision, gpu_name, vram_gb, bottleneck, confidence, peak_vram_gb, step_time_ms
    )

    return diagnostic, fix

# ── Gradio UI Layout ──
with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# ⚡ GemmaOpt: Predictive GPU Profiler")
    gr.Markdown("Identify and fix PyTorch training bottlenecks before you allocate compute.")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Training Configuration")
            model_dd = gr.Dropdown(choices=["bert_base", "llama_3_8b", "resnet50"], value="bert_base", label="Model Architecture")
            gpu_dd = gr.Dropdown(choices=["NVIDIA T4", "NVIDIA A100", "RTX 4090"], value="NVIDIA T4", label="Hardware Target")
            batch_slider = gr.Slider(minimum=1, maximum=1024, step=1, value=512, label="Batch Size")
            precision_radio = gr.Radio(choices=["FP32", "FP16", "BF16"], value="FP32", label="Precision")
            analyze_btn = gr.Button("Run Pre-Flight Check", variant="primary")
            
        with gr.Column(scale=2):
            gr.Markdown("### Diagnostic Telemetry")
            telemetry_box = gr.Textbox(label="XGBoost Prediction", lines=3, interactive=False)
            
            gr.Markdown("### Recommended Implementation")
            code_box = gr.Markdown("Waiting for configuration...")

    # Wire the button to the real backend functions
    analyze_btn.click(
        fn=process_dashboard,
        inputs=[model_dd, batch_slider, precision_radio, gpu_dd],
        outputs=[telemetry_box, code_box]
    )

demo.launch(share=True)

/tmp/ipykernel_55/222462945.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0189c9bedff7dfed52.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
